# Cookie Cats A/B test: does moving the first gate hurt retention?

Cookie Cats is a mobile puzzle game. In this test, players were randomly put into one of two groups when they installed the game: `gate_30` (the first gate stays at level 30) or `gate_40` (the gate moves to level 40). I compare day 1 and day 7 retention between the groups.

Data: [Mobile Games A/B Testing, Cookie Cats](https://www.kaggle.com/datasets/yufengsui/mobile-games-ab-testing) on Kaggle, 90,189 players. Download `cookie_cats.csv` and put it next to this notebook to run it.

In [ ]:
import pandas as pd
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv('cookie_cats.csv')

## Retention overall and by group

In [ ]:
df['retention_1'].mean()

np.float64(0.4452095044850259)

In [ ]:
df['retention_7'].mean()

np.float64(0.1860648194347426)

In [ ]:
df.groupby('version')[['retention_1', 'retention_7']].mean()

,retention_1,retention_7
version,,
gate_30,0.448188,0.190201
gate_40,0.442283,0.182000


## SRM check

In [ ]:
observed = df['version'].value_counts()
expected = [len(df) / 2, len(df) / 2]
chisquare(observed, expected)

Power_divergenceResult(statistic=np.float64(6.9024049496058275), pvalue=np.float64(0.008607987810836262))

**SRM check:** gate_30 has 44,700 players and gate_40 has 45,489, a 49.6/50.4 split.
The chi-square test gives p = 0.0086, so the gap is unlikely to be pure chance.

It's a small gap, though, and many teams use a stricter cutoff like 0.001 for this check.
I'd call it borderline. Before trusting the results I'd ask how players were assigned to groups.


## Day 1 retention

In [ ]:
retained_d1 = df.groupby('version')['retention_1'].sum()
nobs = df.groupby('version').size()
proportions_ztest(retained_d1, nobs)

(np.float64(1.7840862247974725), np.float64(0.07440965529691913))

D1 retention: gate_30 is at 44.8% and gate_40 at 44.2%, a gap of 0.6 points.
The z-test gives p = 0.074, which is above 0.05, so the gap could be chance.
That doesn't prove the gate has no effect on D1. It means this test can't show one.
It also makes sense: most players don't reach level 30 on their first day.

## Day 7 retention

In [ ]:
retained_d7 = df.groupby('version')['retention_7'].sum()
proportions_ztest(retained_d7, nobs)

(np.float64(3.164358912748191), np.float64(0.001554249975614329))

D7 retention: gate_30 is at 19.0% and gate_40 at 18.2%, a gap of 0.8 points.
The z-test gives p = 0.0016, well below 0.05, so this gap is unlikely to be chance.
Moving the gate to level 40 lowered D7 retention by about 4% relative to gate_30.
The effect shows up on day 7 and not day 1 because most players reach level 30 or 40
after their first day.

## Decision

Decision: keep the gate at level 30. Two caveats before treating this as final.
The SRM check was borderline, so I'd confirm how players were assigned to groups.
The test also only covers retention. Gates usually affect revenue too, and that
isn't in this data.